In [ ]:
# Mount Google Drive and configure project directories

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/NLP04_Toxic_Comment_Classification')
NOTEBOOK_DIR = PROJECT_ROOT / '02_notebooks' / '00_project_setup'
RESULTS_DIR = NOTEBOOK_DIR / 'results'

# Verify project directories
assert PROJECT_ROOT.is_dir(), "Project directory not found"
assert NOTEBOOK_DIR.is_dir(), "Notebook directory not found"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_ROOT)
print("Notebook directory:", NOTEBOOK_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
# Verify the required project directories

required_dirs = [
    '00_config',
    '01_data/raw',
    '01_data/processed',
    '01_data/splits',
    '01_data/tokenized',
    '02_notebooks',
    '03_src',
    '04_models',
    '05_references',
    '06_reports'
]

missing_dirs = []

for folder in required_dirs:
    path = PROJECT_ROOT / folder

    if path.is_dir():
        print(f"{folder:<25} Available")
    else:
        print(f"{folder:<25} Missing")
        missing_dirs.append(folder)

print("\nTotal directories:", len(required_dirs))
print("Available directories:", len(required_dirs) - len(missing_dirs))
print("Missing directories:", len(missing_dirs))

assert not missing_dirs, f"Missing directories: {missing_dirs}"

In [ ]:
# Verify the original dataset files and their sizes

RAW_DIR = PROJECT_ROOT / '01_data' / 'raw'

required_files = [
    'train.csv',
    'test.csv',
    'test_labels.csv'
]

missing_files = []

for filename in required_files:
    path = RAW_DIR / filename

    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{filename:<25} {size_mb:.2f} MB")
    else:
        print(f"{filename:<25} Missing")
        missing_files.append(filename)

print("\nRequired files:", len(required_files))
print("Available files:", len(required_files) - len(missing_files))
print("Missing files:", len(missing_files))

assert not missing_files, f"Missing dataset files: {missing_files}"

In [ ]:
# Verify dataset columns and record file fingerprints for reproducibility

import pandas as pd
import hashlib

LABELS = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

expected_columns = {
    'train.csv': ['id', 'comment_text'] + LABELS,
    'test.csv': ['id', 'comment_text'],
    'test_labels.csv': ['id'] + LABELS
}

manifest = []

for filename, expected in expected_columns.items():
    path = RAW_DIR / filename

    # Verify column names
    columns = pd.read_csv(path, nrows=0).columns.tolist()
    assert columns == expected, f"Unexpected columns in {filename}"

    # Calculate SHA-256 fingerprint
    sha256 = hashlib.sha256()

    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            sha256.update(chunk)

    manifest.append({
        'filename': filename,
        'size_bytes': path.stat().st_size,
        'columns': len(columns),
        'schema_valid': True,
        'sha256': sha256.hexdigest()
    })

    print(f"{filename}: Schema verified")

# Save dataset manifest
TABLES_DIR = RESULTS_DIR / 'tables'
TABLES_DIR.mkdir(exist_ok=True)

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(TABLES_DIR / 'dataset_manifest.csv', index=False)

display(manifest_df)

print("\nDataset manifest saved successfully.")

In [ ]:
# Record runtime information and software versions for reproducibility

import sys
import platform
import os
import torch
import numpy as np
import sklearn
import importlib.metadata as metadata

packages = [
    'numpy',
    'pandas',
    'scipy',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'torch',
    'transformers',
    'datasets',
    'accelerate',
    'iterative-stratification',
    'lime',
    'shap',
    'pyarrow'
]

environment = {
    'Python': platform.python_version(),
    'Operating system': platform.platform(),
    'CPU cores': os.cpu_count(),
    'CUDA available': torch.cuda.is_available(),
    'GPU': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
}

print("Runtime information\n")

for key, value in environment.items():
    print(f"{key:<22}: {value}")

# Check installed package versions
package_info = []

for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = 'Not installed'

    package_info.append({
        'package': package,
        'version': version
    })

package_df = pd.DataFrame(package_info)

print("\nSoftware environment\n")
display(package_df)

# Save environment information
package_df.to_csv(TABLES_DIR / 'package_versions.csv', index=False)

pd.DataFrame(
    environment.items(),
    columns=['parameter', 'value']
).to_csv(TABLES_DIR / 'runtime_information.csv', index=False)

In [ ]:
# Initialize random seeds for reproducible experiments

import random

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("Random seed:", SEED)
print("Python random seed initialized")
print("NumPy random seed initialized")
print("PyTorch random seed initialized")

In [ ]:
# Define and save the shared project configuration

import json

CONFIG_DIR = PROJECT_ROOT / '00_config'

project_config = {
    "project_name": "Multi-Label Toxic Comment Classification with Explainability",
    "dataset": "Jigsaw Toxic Comment Classification Challenge",
    "seed": SEED,
    "labels": LABELS,
    "num_labels": len(LABELS),

    "data": {
        "raw_dir": "01_data/raw",
        "processed_dir": "01_data/processed",
        "splits_dir": "01_data/splits",
        "tokenized_dir": "01_data/tokenized"
    },

    "split": {
        "train_ratio": 0.80,
        "validation_ratio": 0.20,
        "strategy": "group-aware multilabel stratification",
        "test_source": "official labelled test dataset"
    },

    "sampling": {
        "strategy": "representative multilabel subsampling",
        "initial_subset_size": 60000,
        "status": "provisional"
    },

    "evaluation": {
        "primary_selection_metric": "macro_average_precision",
        "threshold_selection_metric": "per_label_f1",
        "default_probability_threshold": 0.5,
        "default_svm_margin_threshold": 0.0
    },

    "paths": {
        "notebooks": "02_notebooks",
        "source": "03_src",
        "models": "04_models",
        "references": "05_references",
        "reports": "06_reports"
    },

    "output": {
        "figure_format": "png",
        "figure_dpi": 300,
        "table_format": "csv",
        "prediction_format": "parquet"
    }
}

config_path = CONFIG_DIR / 'project_config.json'

# Preserve an existing configuration if it has been updated
if not config_path.exists():
    with open(config_path, 'w') as file:
        json.dump(project_config, file, indent=4)
else:
    with open(config_path, 'r') as file:
        project_config = json.load(file)

print(json.dumps(project_config, indent=4))
print("\nConfiguration available at:", config_path)

In [ ]:
# Initialize a registry to track experiments and their configurations

REGISTRY_PATH = CONFIG_DIR / 'experiment_registry.csv'

registry_columns = [
    'run_id',
    'notebook',
    'model',
    'feature_representation',
    'loss_function',
    'sampling_strategy',
    'training_samples',
    'seed',
    'max_length',
    'config_path',
    'model_path',
    'results_path',
    'validation_macro_ap',
    'validation_macro_f1',
    'training_time_seconds',
    'status',
    'start_time',
    'end_time',
    'notes'
]

# Preserve existing experiment records
if not REGISTRY_PATH.exists():
    registry_df = pd.DataFrame(columns=registry_columns)
    registry_df.to_csv(REGISTRY_PATH, index=False)
else:
    registry_df = pd.read_csv(REGISTRY_PATH)

print("Experiment registry:", REGISTRY_PATH)
print("Tracking fields:", len(registry_df.columns))
print("Registered experiments:", len(registry_df))

In [ ]:
# Record initial methodological decisions and their justifications

DECISION_LOG_PATH = CONFIG_DIR / 'project_decisions.csv'

decisions = [
    {
        'decision_id': 'D001',
        'component': 'Data splitting',
        'decision': 'Group-aware 80:20 train-validation split',
        'justification': 'Preserve sufficient rare-label validation samples and prevent duplicate leakage.',
        'status': 'Planned'
    },
    {
        'decision_id': 'D002',
        'component': 'Test evaluation',
        'decision': 'Use the official labelled test dataset',
        'justification': 'Maintain a separate benchmark for final evaluation.',
        'status': 'Planned'
    },
    {
        'decision_id': 'D003',
        'component': 'Subsampling',
        'decision': 'Representative subset of 60000 training comments',
        'justification': 'Reduce computational cost while preserving label distributions.',
        'status': 'Provisional'
    },
    {
        'decision_id': 'D004',
        'component': 'Class imbalance',
        'decision': 'Compare standard BCE, weighted BCE and focal loss',
        'justification': 'Investigate imbalance-aware learning without naive resampling.',
        'status': 'Planned'
    },
    {
        'decision_id': 'D005',
        'component': 'Model selection',
        'decision': 'Validation Macro Average Precision',
        'justification': 'Evaluate ranking performance across equally weighted labels under class imbalance.',
        'status': 'Planned'
    },
    {
        'decision_id': 'D006',
        'component': 'Threshold optimization',
        'decision': 'Validation-based per-label F1 optimization',
        'justification': 'Account for different label frequencies and decision boundaries.',
        'status': 'Planned'
    },
    {
        'decision_id': 'D007',
        'component': 'Maximum token length',
        'decision': 'Determine after tokenizer length analysis',
        'justification': 'Balance information retention, truncation and computational cost.',
        'status': 'Pending'
    },
    {
        'decision_id': 'D008',
        'component': 'Reproducibility',
        'decision': 'Fixed initial random seed of 42',
        'justification': 'Maintain consistent experimental conditions across model comparisons.',
        'status': 'Implemented'
    }
]

# Preserve existing decisions and subsequent updates
if not DECISION_LOG_PATH.exists():
    decisions_df = pd.DataFrame(decisions)
    decisions_df.to_csv(DECISION_LOG_PATH, index=False)
else:
    decisions_df = pd.read_csv(DECISION_LOG_PATH)

display(decisions_df)

print("\nDecision log:", DECISION_LOG_PATH)
print("Recorded decisions:", len(decisions_df))

In [ ]:
# Verify setup artifacts and save the notebook summary

required_artifacts = [
    TABLES_DIR / 'dataset_manifest.csv',
    TABLES_DIR / 'package_versions.csv',
    TABLES_DIR / 'runtime_information.csv',
    CONFIG_DIR / 'project_config.json',
    CONFIG_DIR / 'experiment_registry.csv',
    CONFIG_DIR / 'project_decisions.csv'
]

verification = []

for path in required_artifacts:
    verification.append({
        'file': path.name,
        'path': str(path.relative_to(PROJECT_ROOT)),
        'status': 'Available' if path.is_file() else 'Missing'
    })

verification_df = pd.DataFrame(verification)
display(verification_df)

assert verification_df['status'].eq('Available').all(), "Setup incomplete"

verification_df.to_csv(
    TABLES_DIR / 'setup_verification.csv', index=False
)

# Save setup summary
summary = """# Notebook 00 - Project Setup

## Completed Tasks
- Google Drive mounted and project directories verified.
- Raw dataset schemas verified and SHA-256 hashes recorded.
- Runtime and software versions documented.
- Random seed initialized.
- Global project configuration created.
- Experiment registry initialized.
- Methodological decisions recorded.

## Pending Dependencies
- iterative-stratification: Data preparation.
- lime: Explainability, if selected.

## Conclusion
The project environment is ready for exploratory data analysis.
No data preprocessing, splitting or model training was performed.

## Next Stage
Notebook 01 - Dataset EDA.
"""

(RESULTS_DIR / 'summary.md').write_text(summary, encoding='utf-8')

print("\nAll required setup artifacts are available.")
print("Project setup completed.")
print("Summary saved at:", RESULTS_DIR / 'summary.md')